Determine Token Length for Suricata Rules using `Microsoft CodeBERT`

In [ ]:
%pip install numpy transformers

In [ ]:
from transformers import AutoTokenizer
import numpy as np
import os

# Updated to match your training script
MODEL_CHECKPOINT = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def analyze_rule_lengths(rules_list):
    """Calculate precise token counts and data retention for various thresholds."""
    total_token_counts = []
    
    print(f"Analyzing {len(rules_list):,} rules using {MODEL_CHECKPOINT} tokenizer...")
    
    for rule in rules_list:
        # For sequence classification, we just encode the text.
        # add_special_tokens=True ensures we count the <s> and </s> tokens.
        tokens = tokenizer.encode(rule, add_special_tokens=True)
        total_token_counts.append(len(tokens))
    
    total_token_counts = np.array(total_token_counts)
    total_rules = len(total_token_counts)
    
    # Statistical Metrics
    p95 = np.percentile(total_token_counts, 95)
    
    # Retention Calculations
    retention_128 = (np.sum(total_token_counts <= 128) / total_rules) * 100
    retention_256 = (np.sum(total_token_counts <= 256) / total_rules) * 100
    retention_512 = (np.sum(total_token_counts <= 512) / total_rules) * 100
    
    # Recommended sequence length (rounding up to nearest power of 2 from 95th percentile)
    final_rec = 2**(int(p95 - 1).bit_length())
    if final_rec < 128: final_rec = 128
    if final_rec > 512: final_rec = 512 # CodeBERT maxes out at 512

    print("\n" + "="*55)
    print(f"   SURICATA CODEBERT TOKEN ANALYSIS")
    print("="*55)
    print(f"Average token length:      {np.mean(total_token_counts):.1f} tokens")
    print(f"Max token length:          {np.max(total_token_counts)} tokens")
    print("-" * 55)
    print(f"90% of samples are under:  {np.percentile(total_token_counts, 90):.1f} tokens")
    print(f"95% of samples are under:  {p95:.1f} tokens")
    print(f"99% of samples are under:  {np.percentile(total_token_counts, 99):.1f} tokens")
    print("-" * 55)
    print(f"RETENTION METRICS:")
    print(f"  At 128 tokens:           {retention_128:.2f}% ({np.sum(total_token_counts <= 128):,} rules)")
    print(f"  At 256 tokens (Current): {retention_256:.2f}% ({np.sum(total_token_counts <= 256):,} rules)")
    print(f"  At 512 tokens (Max):     {retention_512:.2f}% ({np.sum(total_token_counts <= 512):,} rules)")
    print("-" * 55)
    print(f"RECOMMENDED MAX_LENGTH:    {final_rec}")
    print(f"Estimated retention:       {(np.sum(total_token_counts <= final_rec) / total_rules) * 100:.2f}%")
    print("="*55)
    
    return final_rec

def load_rules(file_paths):
    all_rules = []
    for path in file_paths:
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                # Mirroring your dataset prep logic (skipping comments/empty lines)
                rules = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]
                print(f"Loaded {len(rules):,} rules from {os.path.basename(path)}")
                all_rules.extend(rules)
        except FileNotFoundError:
            print(f"Warning: File not found: {path}")
    return all_rules

# --- EXECUTION ---
# Using the paths from your dataset preparation code
file_paths = [
    '../training-data/dataset_rule_check/good.rules',
    '../training-data/dataset_rule_check/bad.rules'
]

combined_rules = load_rules(file_paths)

if combined_rules:
    analyze_rule_lengths(combined_rules)


Preprocessing and Dataset Preparation

In [ ]:
%pip install pandas scikit-learn

In [ ]:
import json
import random
from sklearn.model_selection import train_test_split
from pathlib import Path
from transformers import AutoTokenizer

# Initialize the tokenizer once
TOKENIZER_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
MAX_THRESHOLD = 512

def prepare_data(good_rules_path, bad_rules_path, output_dir=None):
    data = []
    # Set output directory to current if not provided
    save_path = Path(good_rules_path).parent if output_dir is None else Path(output_dir)
    
    stats = {"good_kept": 0, "good_skipped": 0, "bad_kept": 0, "bad_skipped": 0}

    def process_file(file_path, label, category_prefix):
        print(f"Processing {file_path}...")
        count_kept = 0
        count_skipped = 0
        
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    line = line.strip()
                    # Skip comments and empty lines
                    if line and not line.startswith('#'):
                        # Calculate precise token length
                        # add_special_tokens=True accounts for <s> and </s>
                        tokens = tokenizer.encode(line, add_special_tokens=True)
                        
                        if len(tokens) <= MAX_THRESHOLD:
                            data.append({"text": line, "label": label})
                            count_kept += 1
                        else:
                            count_skipped += 1
                            
            stats[f"{category_prefix}_kept"] = count_kept
            stats[f"{category_prefix}_skipped"] = count_skipped
            
        except FileNotFoundError:
            print(f"Error: File {file_path} not found.")

    # 1. Load Good Rules (Label: 1)
    process_file(good_rules_path, 1, "good")
    
    # 2. Load Bad Rules (Label: 0)
    process_file(bad_rules_path, 0, "bad")

    print("\n" + "="*40)
    print(f"FILTERING STATISTICS (<= {MAX_THRESHOLD} Tokens)")
    print("="*40)
    print(f"Good Rules: Kept {stats['good_kept']:,} | Skipped {stats['good_skipped']:,}")
    print(f"Bad Rules:  Kept {stats['bad_kept']:,} | Skipped {stats['bad_skipped']:,}")
    print(f"Total records for training: {len(data):,}")
    print("-" * 40)

    if not data:
        print(f"Warning: No records matched the <= {MAX_THRESHOLD} token criteria. Files will not be saved.")
        return

    # 3. Shuffle and Split (80% Train, 20% Validation)
    train_data, val_data = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)
    
    # 4. Save to JSONL
    def save_jsonl(data_list, filename):
        full_path = save_path / filename
        with open(full_path, 'w', encoding='utf-8') as f:
            for entry in data_list:
                f.write(json.dumps(entry) + '\n')
        print(f"Saved {len(data_list):,} records to {full_path}")

    save_jsonl(train_data, "train.jsonl")
    save_jsonl(val_data, "validation.jsonl")

# Usage
prepare_data('../training-data/dataset_rule_check/good.rules', '../training-data/dataset_rule_check/bad.rules')

Upload Suricata Training Data

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("No file uploaded.")

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

BERT SLM Training

In [ ]:
%pip install -q transformers datasets accelerate peft scikit-learn evaluate

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import os

# --- CONFIGURATION ---
# MODEL_CHECKPOINT = "distilroberta-base" # Lighter/Faster than bert-base
MODEL_CHECKPOINT = "microsoft/codebert-base"

PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION = 4
MAX_LENGTH = 512    # Hard cap, but dynamic padding prevents wasting compute
LR = 2e-4           # LoRA typically needs higher LR than full fine-tuning (usually 1e-5 to 5e-5)
EPOCHS = 3

# --- 1. LOAD DATASET ---
dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "validation.jsonl"})

# --- 2. TOKENIZATION ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    # OPTIMIZATION: Removed padding="max_length" to allow dynamic padding per batch
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

# OPTIMIZATION: Added num_proc for faster dataset mapping
tokenized_datasets = dataset.map(preprocess_function, batched=True, num_proc=2)

# --- 3. MODEL SETUP (With LoRA) ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "INVALID", 1: "VALID"},
    label2id={"INVALID": 0, "VALID": 1}
)

# Define LoRA Config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # Sequence Classification
    inference_mode=False,
    r=16,               # Gives the model more parameters to learn edge cases
    lora_alpha=32,      # Usually 2x the rank
    # explicitly target all attention and dense layers in RoBERTa/CodeBERT
    target_modules=["query", "key", "value"],
    # OPTIMIZATION: Removed 'embeddings' to save VRAM and speed up training.
    # 'classifier' is sufficient for the classification head in RoBERTa models.
    modules_to_save=["classifier"],
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# --- 4. METRICS ---
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels)
    return {**acc, **f1_score}

# --- 5. TRAINING ARGUMENTS ---
training_args = TrainingArguments(
    output_dir="./suricata-classifier-lora",
    learning_rate=LR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    gradient_checkpointing=True, # Massive VRAM saver: recomputes activations instead of storing them
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",

    # --- NEW OPTIMIZATIONS ---
    optim="adamw_torch_fused",    # Faster optimizer implementation for T4
    group_by_length=False,         # Groups similar length sequences to minimize padding waste
    lr_scheduler_type="cosine",   # Better convergence than linear
    warmup_steps=200,             # warmup_ratio deprecated
    dataloader_num_workers=2      # Uses Colab's CPU cores for faster data loading
)

# --- 6. TRAINER ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    # DataCollatorWithPadding natively handles dynamic padding
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# --- 7. START TRAINING ---
print("Starting optimized training...")
trainer.train()

# --- 8. SAVE MODEL ---
model.save_pretrained("./bert_suricata_model")
tokenizer.save_pretrained("./bert_suricata_model")
print("Model saved to ./bert_suricata_model")

CodeBERT Testing in Google Collab

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load Base Model & Tokenizer
base_model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2)

# Load Trained Adapters
model = PeftModel.from_pretrained(base_model, "./bert_suricata_model")
model.to("cuda")

def check_rule(rule_text):
    inputs = tokenizer(rule_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=1).item()
    return "VALID" if prediction == 1 else "INVALID"

# Test
print(check_rule("alert tcp any any -> any 80 (msg:'Test Rule'; sid:1000001; rev:1;)"))

Running Trained Model Locally

In [ ]:
%pip install torch torchvision torchaudio transformers peft accelerate

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. Determine the best available device (MPS for Mac, CUDA for PC, CPU as fallback)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple M3 Max GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU (CUDA)")
else:
    device = torch.device("cpu")
    print("Using CPU")

# 2. Load Base Model & Tokenizer 
# Note: Ensure this matches what you actually used for training (CodeBERT vs DistilRoBERTa)
base_model_name = "microsoft/codebert-base" 
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2)

# 3. Load Trained Adapters
# Ensure the folder "../slm/bert_suricata_google_collab" is in your local directory
model = PeftModel.from_pretrained(base_model, "../slm/bert_suricata_google_collab")
model.to(device)
model.eval() # Set to evaluation mode

def check_rule(rule_text):
    # Move inputs to the correct device (MPS)
    inputs = tokenizer(rule_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    prediction = torch.argmax(logits, dim=1).item()
    return "VALID" if prediction == 1 else "INVALID"

# Test
rule = 'reject modbus !any any -> any any (msg:"SURICATA Modbus invalid Length"; app-layer-event:modbus.invalid_length; sid:2250003; rev:2;)'
print(f"Prediction: {check_rule(rule)}")

Locally Execute all Suricata rules

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

# --- MAC CONFIGURATION ---
INPUT_FILE = "../training-data/bad.rules"
WRITE_OUTPUT = False  # Set to True to write valid.rules and invalid.rules files
MODEL_PATH = "../slm/bert_suricata_google_collab" # Ensure this folder is in the same directory

# 1. Device Selection (Optimized for M3 Max)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Using Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU (CUDA)")
else:
    device = torch.device("cpu")
    print("Using CPU")

# 2. Load Base Model & Tokenizer
base_model_name = "microsoft/codebert-base"
print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2)

# 3. Load Trained Adapters
inference_model = PeftModel.from_pretrained(base_model, MODEL_PATH)
inference_model.to(device)
inference_model.eval()

def check_rule(rule_text):
    inputs = tokenizer(rule_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = inference_model(**inputs).logits
    prediction = torch.argmax(logits, dim=1).item()
    return "VALID" if prediction == 1 else "INVALID"

def process_rules_file(input_file):
    """Read rules from input file and classify them as valid or invalid."""
    valid_rules = []
    invalid_rules = []

    try:
        with open(input_file, 'r') as f:
            for line in f:
                rule = line.strip()
                # Skip empty lines and comments
                if not rule or rule.startswith('#'):
                    continue

                status = check_rule(rule)
                if status == "VALID":
                    valid_rules.append(rule)
                else:
                    invalid_rules.append(rule)
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.")
        return [], []

    return valid_rules, invalid_rules

def main():
    print(f"Processing rules from '{INPUT_FILE}'...")
    valid_rules, invalid_rules = process_rules_file(INPUT_FILE)

    # Output valid rules section
    print("\n" + "=" * 60)
    print("VALID RULES")
    print("=" * 60)
    if valid_rules:
        for rule in valid_rules:
            print(rule)
    else:
        print("No valid rules found.")

    # Output invalid rules section
    print("\n" + "=" * 60)
    print("INVALID RULES")
    print("=" * 60)
    if invalid_rules:
        for rule in invalid_rules:
            print(rule)
    else:
        print("No invalid rules found.")

    # Summary
    print("\n" + "=" * 60)
    print(f"Summary: {len(valid_rules)} valid, {len(invalid_rules)} invalid")
    print("=" * 60)

    # Write to files if WRITE_OUTPUT is enabled
    if WRITE_OUTPUT:
        with open("valid.rules", 'w') as f:
            for rule in valid_rules:
                f.write(rule + '\n')
        print(f"\nValid rules written to 'valid.rules'")

        with open("invalid.rules", 'w') as f:
            for rule in invalid_rules:
                f.write(rule + '\n')
        print(f"Invalid rules written to 'invalid.rules'")

if __name__ == "__main__":
    main()

Model Evaluation and Accuracy

In [ ]:
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
from collections import defaultdict
import random

# --- CONFIGURATION ---
BASE_MODEL_NAME = "microsoft/codebert-base"
ADAPTER_PATH = "../slm/bert_suricata_google_collab"
MAX_LENGTH = 512
BATCH_SIZE = 32

# Read entire dataset when set to None, otherwise limits to this number of records per source
MAX_RECORD_LIMIT = 5000

# Define your data sources here
DATA_SOURCES = [
    {"path": "../training-data/dataset_rule_check/good.rules", "label": 1, "category": "Valid Dataset"},
    {"path": "../training-data/sample_rules/valid_online.rules", "label": 1, "category": "Online Valid"},
    {"path": "../training-data/sample_rules/valid_generated.rules", "label": 1, "category": "Generated Valid"},
    {"path": "../training-data/dataset_rule_check/bad.rules", "label": 0, "category": "Invalid Dataset"},
    {"path": "../training-data/sample_rules/invalid_aws.rules", "label": 0, "category": "AWS Invalid"},
    {"path": "../training-data/sample_rules/invalid_corrupted.rules", "label": 0, "category": "Corrupted Invalid"},
    {"path": "../training-data/sample_rules/invalid_random.rules", "label": 0, "category": "Random Invalid"},
]

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. LOAD MODEL AND TOKENIZER ---
print("Loading base model and adapters...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME, 
    num_labels=2,
    id2label={0: "INVALID", 1: "VALID"},
    label2id={"INVALID": 0, "VALID": 1}
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to(device)
model.eval()

# --- 2. LOAD & FILTER DATA ---
def load_and_filter_data(sources):
    all_data = [] 
    
    for source in sources:
        count = 0
        skipped_tokens = 0
        path = Path(source["path"])
        
        if not path.exists():
            print(f"Warning: Could not find {path}")
            continue

        # Pass 1: Get the total count of valid lines (excluding comments/empty)
        # We use a simple generator to avoid the tell() error
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            valid_indices = [i for i, line in enumerate(f) 
                             if line.strip() and not line.startswith('#')]
            total_valid_lines = len(valid_indices)

        # Determine random start index within the list of valid lines
        start_idx = 0
        if MAX_RECORD_LIMIT and total_valid_lines > MAX_RECORD_LIMIT:
            # We pick a random starting point in our valid_indices list
            start_idx = random.randint(0, total_valid_lines - MAX_RECORD_LIMIT)
        
        # Pass 2: Re-read the file and pick only the lines in our random window
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            current_valid_count = 0
            
            for line in f:
                line_content = line.strip()
                if not line_content or line_content.startswith('#'):
                    continue
                
                # Check if this valid line falls within our random window
                if start_idx <= current_valid_count < (start_idx + MAX_RECORD_LIMIT):
                    # TOKEN CHECK (Strict 512 Limit)
                    tokens = tokenizer.encode(line_content, add_special_tokens=True)
                    
                    if len(tokens) <= MAX_LENGTH:
                        all_data.append({
                            "text": line_content,
                            "label": source["label"],
                            "category": source["category"]
                        })
                        count += 1
                    else:
                        skipped_tokens += 1
                
                # Move to next valid line index
                current_valid_count += 1
                
                # Break early if we have reached our quota for this file
                if count >= MAX_RECORD_LIMIT:
                    break
                        
            print(f"Loaded {count} rules from {source['category']} (Started at valid line {start_idx}, Skipped {skipped_tokens} oversized)")
            
    return all_data

print("Filtering and loading datasets...")
test_dataset = load_and_filter_data(DATA_SOURCES)

if not test_dataset:
    raise ValueError("No test data found. Check your file paths and token limits.")

# --- 3. INFERENCE LOOP ---
print(f"Running inference on {len(test_dataset)} samples...")
all_texts = [d['text'] for d in test_dataset]
all_labels = [d['label'] for d in test_dataset]
all_preds = []

for i in tqdm(range(0, len(all_texts), BATCH_SIZE)):
    batch_texts = all_texts[i:i + BATCH_SIZE]
    inputs = tokenizer(
        batch_texts, 
        padding=True, 
        truncation=True, 
        max_length=MAX_LENGTH, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        all_preds.extend(batch_preds)

# Attach predictions back to the dataset objects
for i in range(len(test_dataset)):
    test_dataset[i]['pred'] = all_preds[i]

# --- 4. CALCULATE METRICS ---
print("\n" + "="*60)
print("GLOBAL EVALUATION RESULTS")
print("="*60)
print(f"Overall Accuracy: {accuracy_score(all_labels, all_preds) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["INVALID", "VALID"]))

print("\n" + "="*60)
print("CATEGORY-SPECIFIC STATISTICS")
print("="*60)

# Group results by category
stats = defaultdict(lambda: {"correct": 0, "total": 0})
for item in test_dataset:
    cat = item['category']
    stats[cat]["total"] += 1
    if item['label'] == item['pred']:
        stats[cat]["correct"] += 1

# Display table
print(f"{'Category':<25} | {'Accuracy':<10} | {'Count':<10}")
print("-" * 50)
for cat, data in stats.items():
    acc = (data["correct"] / data["total"]) * 100
    print(f"{cat:<25} | {acc:>8.2f}% | {data['total']:<10}")

print("="*60)

DeepEval Evaluation

In [ ]:
%pip install -U deepeval transformers torch ollama ipywidgets
!ollama pull qwen2.5-coder:7b
!ollama list

In [ ]:
import asyncio

import torch
import random
import time
import gc
import os
from tqdm import tqdm
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.evaluate import AsyncConfig, DisplayConfig, CacheConfig
from deepeval import evaluate
from peft import PeftModel
from ollama import AsyncClient
import ollama

# --- 1. Configuration & Data Sources ---
DATA_SOURCES = [
    {"path": "../training-data/dataset_rule_check/good.rules", "label": 1, "category": "Valid Dataset"},
    {"path": "../training-data/sample_rules/valid_online.rules", "label": 1, "category": "Online Valid"},
    {"path": "../training-data/sample_rules/valid_generated.rules", "label": 1, "category": "Generated Valid"},
    {"path": "../training-data/dataset_rule_check/bad.rules", "label": 0, "category": "Invalid Dataset"},
    {"path": "../training-data/sample_rules/invalid_aws.rules", "label": 0, "category": "AWS Invalid"},
    {"path": "../training-data/sample_rules/invalid_corrupted.rules", "label": 0, "category": "Corrupted Invalid"},
    {"path": "../training-data/sample_rules/invalid_random.rules", "label": 0, "category": "Random Invalid"},
]

# Comprehensive Suricata Protocol Specs mapped for LLM Context Context
PROTOCOL_SPECS = {
    "http": ["Uses http.method, http.uri, http.request_body, http.header.", "Often paired with nocase, distance, within."],
    "ftp": ["Uses ftp.command, ftp.data."],
    "tls": ["Uses tls.sni, tls.cert_subject, tls.cert_issuer, tls.version.", "Check for proper handshake flow."],
    "smb": ["Uses smb.tree, smb.filename, smb.named_pipe, smb.share."],
    "dns": ["Uses dns.query, dns.opcode, dns.flags."],
    "dcerpc": ["Uses dcerpc.iface, dcerpc.opnum, dcerpc.req_call."],
    "dhcp": ["Uses dhcp.type, dhcp.client_mac, dhcp.hostname."],
    "ssh": ["Uses ssh.software, ssh.softwareversion, ssh.protoversion."],
    "smtp": ["Uses smtp.helo, smtp.mail_from, smtp.rcpt_to, smtp.subject."],
    "imap": ["Uses imap.request, imap.response."],
    "nfs": ["Uses nfs.procedure, nfs.version."],
    "ike": ["Uses ike.vendor, ike.exch_type, ike.payload_type."],
    "krb5": ["Uses krb5.cname, krb5.sname, krb5.msg_type."],
    "ntp": ["Uses ntp.mode, ntp.stratum."],
    "rfb": ["Uses rfb.sec_type, rfb.version."],
    "rdp": ["Uses rdp.cookie, rdp.name."],
    "snmp": ["Uses snmp.community, snmp.pdu_type, snmp.version."],
    "tftp": ["Uses tftp.opcode, tftp.filename."],
    "quic": ["Uses quic.sni, quic.version, quic.cyu."],
    "mqtt": ["Uses mqtt.topic, mqtt.message, mqtt.type, mqtt.qos."]
}

# --- 2. Optimized Judge Setup ---
# 1. Create a Fake OpenAI Response Wrapper to satisfy DeepEval's internal parser
class MockMessage:
    def __init__(self, content):
        self.content = content

class MockChoice:
    def __init__(self, content):
        self.message = MockMessage(content)

class MockResponse:
    def __init__(self, content):
        self.choices = [MockChoice(content)]

class OllamaJudge(DeepEvalBaseLLM):
    def __init__(self, model_name="qwen2.5-coder:7b"):
        self.model_name = model_name
        
    def load_model(self): 
        return self.model_name
        
    def generate(self, prompt: str) -> str:
        time.sleep(0.5) # Prevent socket exhaustion
        # Note: We removed the hardcoded 'score/reason' suffix. 
        # DeepEval will provide its own instructions in the 'prompt' variable.
        response = ollama.generate(
            model=self.model_name, 
            prompt=prompt,
            format="json" 
        )
        return response['response']

    async def a_generate(self, prompt: str) -> str:
        await asyncio.sleep(0.5) 
        client = AsyncClient(timeout=300.0)
        response = await client.generate(
            model=self.model_name, 
            prompt=prompt,
            format="json"
        )
        return response['response']

    def generate_raw_response(self, prompt: str, **kwargs):
        res_str = self.generate(prompt)
        return MockResponse(res_str), 0.0 

    async def a_generate_raw_response(self, prompt: str, **kwargs):
        res_str = await self.a_generate(prompt)
        return MockResponse(res_str), 0.0 

    def get_model_name(self): 
        return f"Ollama ({self.model_name})"

# Hardware Optimization for M3 Max
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
local_judge = OllamaJudge()

tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
base_model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).to(device)
model = PeftModel.from_pretrained(base_model, "../slm/bert_suricata_google_collab_3").to(device)
model.eval()

def run_clean_eval(data_sources, total_sample_per_source=10, batch_size=5):
    all_test_cases = []
    
    print(f"🧠 Step 1: Running CodeBERT Inference on {len(data_sources)} sources...")

    for source in data_sources:
        file_path, expected_label, category = source['path'], source['label'], source['category']
        if not os.path.exists(file_path): continue

        with open(file_path, 'r') as f:
            all_rules = [line.strip() for line in f if line.strip() and not line.startswith("#")]
        
        sampled_rules = random.sample(all_rules, min(total_sample_per_source, len(all_rules)))
        
        for i in range(0, len(sampled_rules), batch_size):
            batch = sampled_rules[i : i + batch_size]
            
            inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                outputs = model(**inputs)
                predictions = torch.argmax(outputs.logits, dim=-1).tolist()

            for rule, pred in zip(batch, predictions):
                # Smarter protocol matching
                rule_lower = rule.lower()
                matched_protocols = [p for p in PROTOCOL_SPECS.keys() if f" {p} " in rule_lower or f" {p}." in rule_lower]
                protocol = matched_protocols[0] if matched_protocols else "http" # Default fallback
                
                context_data = [f"Protocol: {protocol}"] + PROTOCOL_SPECS.get(protocol, [])
                
                test_case = LLMTestCase(
                    input=rule,
                    actual_output=f"Label {pred}",
                    expected_output=f"Label {expected_label}",
                    retrieval_context=context_data, # Required for RAG metrics (Contextual Precision/Faithfulness)
                    context=context_data,
                    additional_metadata={"category": category, "correct": pred == expected_label}
                )
                all_test_cases.append(test_case)

            del inputs, outputs
            if torch.backends.mps.is_available(): torch.mps.empty_cache()
            gc.collect()

    # --- 3. Define Metrics (CRITICAL FIX: Use local_judge, NOT model) ---
    correctness_metric = GEval(
        name="AWS-Suricata Compliance Auditor",
        model=local_judge,
        evaluation_steps=[
            "Check if the Suricata rule follows the standard action-protocol-address format.",
            "Verify that all options and sticky buffers (like http.uri or tls.sni) are valid and correctly terminated by semicolons.",
            "Determine if the rule is compatible with AWS Network Firewall (RuleGroups).",
            "Compare the CodeBERT Label (Actual Output) with the rule's validity.",
            "Assign a score of 1.0 if the Label correctly identifies validity, otherwise assign 0.0."
        ],
        criteria="""
        Act as a Network Security Engineer. Evaluate if the CodeBERT 'Actual Output' (Label) 
        correctly identifies the rule's validity based on TWO strict criteria:

        1. SURICATA SYNTAX & SEMANTICS (Standard CLI):
        - Must follow: 'action protocol src_ip port -> dst_ip port (options)'.
        - Must use valid 'sticky buffers' (e.g., http.uri, tls.sni) correctly separated by semicolons.
        - PCRE syntax and modifiers (nocase, fast_pattern) must be syntactically placed.

        2. AWS RULEGROUP COMPATIBILITY:
        - Detect if the rule uses AWS-unsupported features (e.g., certain flowbit combinations 
            or non-standard address variables).
        - Note: AWS RuleGroups are often stricter than standard Suricata CLI.
        - A rule is INVALID (Label 0) if it fails Standard Syntax OR AWS Boto3 compatibility.

        SCORING:
        - Score 1.0: The Label (0 or 1) perfectly matches the rule's actual state.
        - Score 0.0: The Label is a False Positive or False Negative based on the criteria above.
        """,
        # We include EXPECTED_OUTPUT so the Judge knows the 'Ground Truth' from your dataset
        evaluation_params=[
            LLMTestCaseParams.INPUT, 
            LLMTestCaseParams.ACTUAL_OUTPUT, 
            LLMTestCaseParams.EXPECTED_OUTPUT,
            LLMTestCaseParams.CONTEXT
        ],
        threshold=0.7
    )

    # --- 4. Single DeepEval Call ---
    print(f"\n⚖️ Step 2: Starting LLM Judge Evaluation ({len(all_test_cases)} cases)...")

    results = evaluate(
        test_cases=all_test_cases,
        metrics=[correctness_metric],
        display_config=DisplayConfig(show_indicator=True, print_results=False, verbose_mode=False),
        cache_config=CacheConfig(use_cache=True),
        async_config=AsyncConfig(run_async=False)
    )

    # --- 5. Final Summary ---
    print("\n" + "="*50)
    print(f"{'CATEGORY':<25} | {'BERT ACCURACY':<15}")
    print("-" * 50)

    for source in data_sources:
        cat = source['category']
        cat_cases = [tc for tc in all_test_cases if tc.additional_metadata['category'] == cat]
        if not cat_cases: continue
        
        correct_count = sum(1 for tc in cat_cases if tc.additional_metadata['correct'])
        accuracy = (correct_count / len(cat_cases)) * 100
        print(f"{cat:<25} | {accuracy:>8.2f}%")

    print("="*50)

if __name__ == "__main__":
    run_clean_eval(DATA_SOURCES, total_sample_per_source=10, batch_size=5)